In [ ]:
%matplotlib inline
import os
import pandas as pd
import sionna.rt
import matplotlib.pyplot as plt
import mitsuba as mi
import drjit as dr
import numpy as np
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, RadioMapSolver

In [ ]:
# Load the Mitsuba description scene exported from Blender
scene = sionna.rt.load_scene(filename='blender/massy.xml')

In [ ]:
# Configure global antenna arrays for all transmitters (Tx) and receivers (Rx)
# TODO: Upgrade to a more complex AntennaArray setup according to source data
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             pattern="iso",
                             polarization="V")
scene.rx_array = scene.tx_array

In [ ]:
from pyproj import Transformer, CRS

# Define the geographical boundary of the target region
LAT_MAX, LAT_MIN = 48.7409, 48.7171
LON_MIN, LON_MAX = 2.2451, 2.3013

# Calculate the center origin point of the scene
LAT_ORIGIN = (LAT_MAX + LAT_MIN)/2
LON_ORIGIN = (LON_MIN + LON_MAX)/2
ALT_ORIGIN = 0

# Determine the corresponding UTM (Universal Transverse Mercator) zone based on longitude
_ZONE = int((LON_ORIGIN + 180) / 6) + 1
_IS_SOUTH = LAT_ORIGIN < 0
UTM_CRS = CRS.from_dict({
        'proj': 'utm', 
        'zone': _ZONE,
        'south': _IS_SOUTH
    })

def latlon_to_xy_batch(lats, lons):
    """
    Batch convert geographic coordinates (Latitude, Longitude) to the local Cartesian coordinates (x, y) of the scene.
    Supports both NumPy arrays and Pandas Series as inputs.
    """
    # EPSG:4326 corresponds to WGS84 Geodetic Coordinate System
    transformer = Transformer.from_crs("epsg:4326", UTM_CRS, always_xy=True)
    x_origin, y_origin = transformer.transform(LON_ORIGIN, LAT_ORIGIN)
    x_objs, y_objs = transformer.transform(lons, lats)

    # Calculate relative coordinates based on the scene's origin center
    xs = x_objs - x_origin
    ys = y_objs - y_origin
    return np.asarray(xs).astype(np.float32), np.asarray(ys).astype(np.float32)

def get_terrain_z_batch(scene, xs, ys):
    """
    Batch obtain the absolute altitude (z-coordinate) of the terrain surface at given (x, y) coordinates.
    This utilizes Mitsuba's ray tracing kernel to perform vertical intersection tests.
    """
    mi_scene = scene.mi_scene

    # Convert the input coordinates to DrJit arrays for parallel hardware acceleration
    xs_dr = mi.Float(xs)
    ys_dr = mi.Float(ys)
    
    # Create a vertical ray probe: start from a sufficiently high altitude (10,000 meters) and shoot downwards (0, 0, -1)
    ray_origin = mi.Point3f(xs_dr, ys_dr, 10000.0)
    ray_direction = mi.Vector3f(0.0, 0.0, -1.0)
    ray = mi.Ray3f(ray_origin, ray_direction)
    
    # Compute intersection between the downward rays and the scene geometry
    si = mi_scene.ray_intersect(ray)

    if not dr.all(si.is_valid()) or si.p.z is None:
        print("Failed to obtain z")

    # If intersection is valid, extract the z coordinate; otherwise default to 0.0
    highest_z = dr.select(si.is_valid(), si.p.z, 0.0)
    
    return np.array(highest_z)

In [ ]:
DISPLAY_RADIUS=None
RECEIVER_GROUND_HEIGHT=2

def get_tx_name(id):
    return f'tx_{id}'

def get_rx_name(id):
    return f'rx_{id}'

def add_tx(scene, name, position, orientation=(0,0,0)):
    # TODO: Clarify and incorporate the 'power_dbm' parameter configuration here
    tx = Transmitter(name=name, position=position, orientation=orientation, display_radius=DISPLAY_RADIUS)
    scene.add(tx)

def add_txs(scene, df_tx:pd.DataFrame):
    """
    Batch map geographic Tx coordinates into the 3D scene and instantiate Transmitter objects.
    """
    xs, ys = latlon_to_xy_batch(df_tx['Latitude'].values, df_tx['Longitude'].values)
    terrain_zs = get_terrain_z_batch(scene, xs, ys)
    tx_heights = df_tx['height'].values
    final_zs = terrain_zs + tx_heights

    for i, (_, row) in enumerate(df_tx.iterrows()):
        name = get_tx_name(row['ID'])
        
        x = float(xs[i])
        y = float(ys[i])
        z = float(final_zs[i])
        
        # TODO: Implement orientation configuration based on source data
        
        # Debug output to verify transimitter height alignment
        print(f"TX {name} placed at x={x:.2f}, y={y:.2f}, z={z:.2f} (Terrain z={terrain_zs[i]:.2f}, Device height={tx_heights[i]:.2f})")
        
        add_tx(scene, name, (x, y, z))

def add_rx(scene, name, position):
    rx = Receiver(name=name, position=position, display_radius=DISPLAY_RADIUS)
    scene.add(rx)

def add_rxs(scene, df_rx:pd.DataFrame):
    """
    Batch map geographic Rx coordinates into the 3D scene and instantiate Receiver objects.
    """
    xs, ys = latlon_to_xy_batch(df_rx['Latitude'].values, df_rx['Longitude'].values)
    terrain_zs = get_terrain_z_batch(scene, xs, ys)
    final_zs = terrain_zs + RECEIVER_GROUND_HEIGHT

    for i, (_, row) in enumerate(df_rx.iterrows()):
        name = get_rx_name(row['sensor_id'])
        
        x = float(xs[i])
        y = float(ys[i])
        z = float(final_zs[i])
        
        # Debug output to verify sensor height alignment
        print(f"Sensor {name} placed at x={x:.2f}, y={y:.2f}, z={z:.2f} (Terrain z={terrain_zs[i]:.2f})")
        
        add_rx(scene, name, (x, y, z))

In [ ]:
def set_frequence(scene, frequence, unit:str):
    """
    Utility function to configure the carrier frequency of the simulation scene.
    """
    if unit.lower() == 'mhz':
        scene.frequency = frequence * 1e6
    elif unit.lower() == 'ghz':
        scene.frequency = frequence * 1e9
    else:
        raise ValueError(f"Unsupported frequency unit: {unit}")

In [ ]:
# Load the preprocessed transmitter and receiver information from files
transimitter_file_path = os.path.join('data','transimitters', '3500_mhz.csv')
receive_file_path = os.path.join('data', 'sensor_location.csv')

df_tx = pd.read_csv(transimitter_file_path, encoding='utf-8-sig')
df_rx = pd.read_csv(receive_file_path, encoding='utf-8')

In [ ]:
# TODO: Some ITU materials are currently undefined at specific custom frequencies.
# Therefore, we temporarily fall back to Sionna's default band setup.
# Once parameters are complete, uncomment the line below to fully enable custom frequencies:
# set_frequence(scene, 700, 'mhz')
add_txs(scene, df_tx)
add_rxs(scene, df_rx)

In [ ]:
# Approach 1: Compute Radio Map on a flat bounding grid solver

rm_solver = RadioMapSolver()

rm = rm_solver(scene,
    max_depth=5,           # Maximum number of ray-scene interactions
    samples_per_tx=10**7 , # Total Monte Carlo ray samples per Tx (higher = less noise, more memory)
    cell_size=(5, 5),      # Spatial resolution of each radio map grid cell (in meters)
    center=[0, 0, 0],      # Coordinate center point of the target radio map grid
    size=[5000, 5000],     # Total dimensions (width, height, in meters) of the radio map area
    orientation=[0, 0, 0]) # Spatial orientation of the evaluation grid planes


In [ ]:
# Visualize the calculated propagation metrics
rm.show(metric="path_gain")

# Visualize the Received Signal Strength (RSS)
rm.show(metric="rss")

# Visualize the Signal-to-Interference-plus-Noise Ratio (SINR)
rm.show(metric="sinr")

In [ ]:
# Approach 2: Compute Radio Map adaptively conforming to the continuous 3D terrain surface mesh

rm_solver = RadioMapSolver()

# Extract and duplicate the terrain surface as the continuous measuring mesh target
mesurement_surface = scene.objects["terrain"].clone(as_mesh=True)

rm = rm_solver(scene,
    measurement_surface=mesurement_surface,
    samples_per_tx=10**8,   # Increased ray counts for complex surface conformal calculations
    max_depth=5)

In [ ]:
# Debug Block: Manually verify single arbitrary coordinate mapping and placement
xs, ys = latlon_to_xy_batch(48.72306, 2.28583)
add_tx(scene, 'tx', (xs, ys, 100))

In [ ]:
# Debug Block: Clean up the temporary test node from the simulation scene
scene.remove('tx')

In [ ]:
# Render interactive previews of the 3D scene topology
scene.preview()

In [ ]:
# Preview the 3D scene overlaid with the computed Radio Map coverage
scene.preview(radio_map=rm, rm_vmin=-200)

In [ ]:
# Debug Block: Inspect and verify all loaded objects within the active Sionna scene structure
scene.objects

In [ ]:
# Debug Block: Verify electromagnatic property bindings of specified objects
scene.get("water").radio_material